# MTRAG Dataset Inspection

::::::{div}
:class: proposal-document

:::::{div}
:class: proposal-header exclude-from-outline

## Opening MTRAG, One Task at a Time

:::{div}
:class: proposal-subtitle

A practical inspection of the conversations, evidence passages and human ratings behind the research proposal.
:::

:::{div}
:class: proposal-meta

Research notebook · Jenny Lee

Inspected October 23, 2026 · Pinned public release
:::

:::{div}
:class: document-actions

{button}`Inspection script <../data/downloads/inspect_mtrag.py>`

{button}`Verified counts <../data/Research/mtrag-inspection-summary.json>`
:::
:::::

:::{dropdown} Sections
:class: proposal-toc

* [Dataset at a glance](#mtrag-overview)
* [Find the right files](#mtrag-files)
* [Open and validate](#mtrag-open)
* [Read one task](#mtrag-task)
* [Inspect the distributions](#mtrag-distributions)
* [Open the human ratings](#mtrag-human)
* [Connect to the research](#mtrag-research)
* [Sources and next steps](#mtrag-sources)
:::

(mtrag-overview)=
## Dataset at a Glance

MTRAG is the main public data resource in the [research proposal](Proposal.ipynb). It lets us examine an answer to a current question while retaining the earlier conversation and its documentary evidence. This page opens the released files, checks their contents and explains what can be carried into a statistical analysis.

::::{grid} 1 1 2 2

:::{grid-item-card} 110 conversations
The grouping units for the analysis. The inspected conversations contribute **3–12 tasks each**, with an average of **7.65**.
:::

:::{grid-item-card} 842 turn-level tasks
Each task ends at one user question. Earlier messages are supplied as fixed history; the current reference answer is separate.
:::

:::{grid-item-card} 4 document collections
Wikipedia, technical documentation, finance and government. Preserve collection identity when comparing results.
:::

:::{grid-item-card} 159 human-rated tasks
A separate subset from **20 conversations**, with **477 task–model evaluations** across three response sources.
:::
::::

These counts were computed from the [pinned IBM files](https://github.com/IBM/mt-rag-benchmark/tree/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human), rather than inferred from the proposal. The original paper rounds the average to 7.7 turns.

:::{admonition} Definition · What is a task?
:class: note

A **conversation** is a sequence of user and agent messages. A **task** is one point in that sequence: the earlier messages, the current user question, supplied passages and a reference answer. A **candidate answer** is a model's response to that task. A **rating** is one evaluator's judgment of one candidate answer on one metric. These are different observational units.
:::

:::{dropdown} MTRAG versus MTRAG-UN

This inspection covers the original **MTRAG human data**, under `mtrag-human/`. The proposal also names **MTRAG-UN**, a separate extension with 666 tasks addressing additional conversational challenges. IBM's current repository positions MTRAG as training data and MTRAG-UN as the evaluation benchmark for its newer setting. Do not silently pool the releases or treat the same conversation as independent training and evaluation data. Record the release and intended experimental role explicitly. See the [repository overview](https://github.com/IBM/mt-rag-benchmark) and [MTRAG-UN paper](https://arxiv.org/abs/2602.23184).
:::

(mtrag-files)=
## Find the Right Files

Start with the **reference generation tasks**. They contain the question, history, passages and target together, so inspecting a task does not require downloading every corpus. The following paths are relative to the [official repository](https://github.com/IBM/mt-rag-benchmark/tree/53fef6f5917c7e10c436f697e69a955dbc795d81).

| File or directory | What is inside | How to open it |
| :--- | :--- | :--- |
| `mtrag-human/conversations/conversations.json` | 110 complete conversations, including collection and generation metadata | `json.load(...)`; top-level list |
| `mtrag-human/generation_tasks/reference.jsonl` | 842 tasks with reference evidence | One JSON object per line |
| `mtrag-human/generation_tasks/RAG.jsonl` | 842 tasks with five retrieved passages each | One JSON object per line |
| `mtrag-human/generation_tasks/reference+RAG.jsonl` | 436 tasks with five passages each | One JSON object per line |
| `mtrag-human/evaluations/reference_subset_with_human_evaluations.json` | Tasks, documents, response sources, metric definitions and ratings | `json.load(...)`; top-level dictionary |
| `corpora/passage_level/` | Four compressed full passage collections | Unzip, then read JSONL |
| `mtrag-human/retrieval_tasks/` | Collection-specific queries and relevance judgments | Query JSONL and qrels TSV |

### Choose the Evidence Setting

| Setting | Tasks checked | Evidence supplied | Interpretation |
| :--- | ---: | :--- | :--- |
| Reference | 842 | 0–9 reference passages | Study generation with the released reference evidence |
| Full RAG | 842 | Five retrieved passages | Retrieval errors can affect the answer |
| Reference + RAG | 436 | Five passages, keeping reference evidence among them | Study generation with distractors on a restricted subset |

The [generation documentation](https://github.com/IBM/mt-rag-benchmark/blob/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human/generation_tasks/README.md) limits Reference + RAG to tasks with at most two reference contexts. **436 is not a random sample of 842.** Compare settings on matched `task_id` values and report which tasks remain.

:::{dropdown} When do I need the full corpus?

For inspecting generation inputs, the task's `contexts` already contains passage text. Download the corpus when rebuilding retrieval, checking a passage against its collection or evaluating retrieval coverage. IBM recommends the **passage-level** release to align with its reference passage boundaries.

| Corpus | Subject | Documents reported by IBM | Passages reported by IBM |
| :--- | :--- | ---: | ---: |
| ClapNQ | Wikipedia | 4,293 | 183,408 |
| Cloud | Technical documentation | 57,638 | 61,022 |
| FiQA | Finance | 7,661 | 49,607 |
| Govt | Government | 8,578 | 72,422 |

These full-corpus counts are from the [corpus README](https://github.com/IBM/mt-rag-benchmark/blob/53fef6f5917c7e10c436f697e69a955dbc795d81/corpora/README.md); the compressed corpora were not independently enumerated for this inspection. Retrieval qrels cover answerable and partial tasks, so their population differs from all 842 generation tasks.
:::

(mtrag-open)=
## Open and Validate

The downloadable inspection script uses **Python 3.9+ and the standard library only**. It downloads five public files at a fixed repository commit, checks identifiers and joins, and writes a summary plus a long-form human-rating table. It makes no generation or judge calls.

```bash
python inspect_mtrag.py --cache ./mtrag-cache
```

The output directory contains the downloaded source files, `inspection-summary.json` and `human-ratings.jsonl`. File sizes and SHA-256 hashes are recorded so a later run can be compared with the snapshot. Use a fresh cache if you have edited the source files.

:::{dropdown} Open the reference file yourself

Run this after the inspection script, from the same working directory:

```python
import json
from pathlib import Path
from collections import Counter

cache = Path("mtrag-cache")
path = cache / "mtrag-human__generation_tasks__reference.jsonl"
with path.open(encoding="utf-8") as stream:
    tasks = [json.loads(line) for line in stream if line.strip()]

by_id = {task["task_id"]: task for task in tasks}
assert len(by_id) == len(tasks) == 842
assert len({t["conversation_id"] for t in tasks}) == 110
assert len({(t["conversation_id"], int(t["turn"]))
            for t in tasks}) == len(tasks)

for task in tasks:
    assert task["input"][-1]["speaker"] == "user"
    assert len(task["input"]) == 2 * int(task["turn"]) - 1

print(Counter(t["Collection"] for t in tasks))
print(Counter(label for t in tasks for label in t["Answerability"]))
```

The last assertion is a property checked for this release, not a universal rule for all conversational datasets. A later release with tool messages could have a different message structure.
:::

:::{admonition} Actual file schema takes priority
:class: important

The README's illustrative schema uses lowercase `collection` and `answerability` and describes an integer turn. In the inspected generation files, the keys are **`Collection`** and **`Answerability`**, and **`turn` is a string**. Use `int(task["turn"])` before sorting. Otherwise turn `"10"` can appear before `"2"`.
:::

(mtrag-task)=
## Read One Task

```{figure} ../data/Research/mtrag-task-flow.svg
:alt: Fixed history, current question and passages feed a candidate answer. Judges score that answer, optionally using the separate reference. All scores preserve the task identity.
:width: 100%

The candidate and the judge have different input requirements. Keep the target answer out of the candidate's generation prompt.
```

| Field in the generation file | Meaning | Inspection question |
| :--- | :--- | :--- |
| `task_id` | Identifier for the current task | Is it unique within this setting? |
| `conversation_id` | Shared conversation identifier | Which tasks must stay in the same split? |
| `turn` | Position of the current user turn | Has it been converted to an integer? |
| `input[:-1]` | Earlier user and agent messages | Which earlier details does the question require? |
| `input[-1]` | Current user question | Is its meaning clear without the history? |
| `contexts` | Evidence available in this setting | Are there passages, and do they support an answer? |
| `targets` | Released reference answer(s) | Is the reference needed by the chosen metric? |
| `Answerability` | List of answerability labels | Should an answer, partial answer or abstention be expected? |
| `Question Type`, `Multi-Turn` | Task descriptors | Which subgroups may behave differently? |
| `Collection` | Source document collection | Is performance being compared across domains? |

### A Real First Turn

The first reference task has ID `dd6b6ffd177f2b311abe676261279d2f<::>1`. This shortened display preserves its actual question, label and empty evidence list; it omits metadata and the target text.

```json
{
  "turn": "1",
  "input": [{
    "speaker": "user",
    "text": "where do the arizona cardinals play this week"
  }],
  "contexts": [],
  "Answerability": ["UNANSWERABLE"]
}
```

The source collection concerns historical material, whereas “this week” requires a current schedule. The released reference declines to answer. This is a useful abstention case: a confident schedule unsupported by the evidence should not receive credit merely for sounding helpful.

:::{dropdown} Follow the same conversation to turn 2

The next task ends with “Do the Arizona Cardinals play outside the US?” It is labeled `ANSWERABLE`. Its `input` contains three messages: the previous question, the previous reference response and the new question.

```python
conversation_id = tasks[0]["conversation_id"]
conversation = sorted(
    (t for t in tasks if t["conversation_id"] == conversation_id),
    key=lambda t: int(t["turn"]),
)
for task in conversation:
    print(int(task["turn"]), task["Answerability"],
          len(task["contexts"]), task["input"][-1]["text"])
```

The earlier answer here comes from the supplied history. It is not automatically replaced by the candidate model's answer from your previous experiment. Generating all tasks independently therefore studies a fixed-history benchmark, rather than a live rollout in which model errors change future context.
:::

:::{admonition} Definition · Reference is not a human quality label
:class: note

A reference answer supplies comparison content. A human quality label evaluates a particular response. The dataset creation involved a model-assisted workflow and human review; reference metadata can still name Mixtral. Do not infer that every reference was written from scratch by a human, or that every reference answer has a perfect human rating.
:::

(mtrag-distributions)=
## Inspect the Distributions

The following tables are computed from the 842 reference tasks. Counts refer to tasks, not independent conversations or individual annotators.

::::{grid} 1 1 2 2

:::{grid-item-card} Domain coverage

| Collection | Tasks |
| :--- | ---: |
| ClapNQ | 224 |
| Cloud | 205 |
| FiQA | 199 |
| Govt | 214 |

The four domains have similar task counts, but similar counts do not imply similar difficulty.
:::

:::{grid-item-card} Answerability

| Label | Tasks |
| :--- | ---: |
| ANSWERABLE | 709 |
| PARTIAL | 68 |
| UNANSWERABLE | 55 |
| CONVERSATIONAL | 10 |

An overall average is dominated by answerable questions. Report abstention behavior separately.
:::
::::

| Structural check | Observed result | Why it matters |
| :--- | :--- | :--- |
| Unique task IDs | 842 of 842 | A within-setting task join is unambiguous |
| Unique conversation–turn pairs | 842 of 842 | Turn identity is preserved |
| History length | `2 × turn − 1` in every task | Current input ends with a user message |
| First turns | 110 | One per conversation |
| Later turns | 732 | Most tasks have shared earlier context |
| Multi-turn labels | 622 Follow-up; 110 Clarification; 110 N/A | Later-turn behavior is central to the resource |
| Empty reference contexts | 65 | All 55 unanswerable and 10 conversational tasks |
| Full RAG contexts | Five for every task | Having retrieved text does not establish answerability |

:::{dropdown} Inspect passage counts and missing evidence

```python
print(Counter(len(t["contexts"]) for t in tasks))
# {0: 65, 1: 105, 2: 266, 3: 243, 4: 96,
#  5: 36, 6: 21, 7: 8, 8: 1, 9: 1}  (shown in sorted order)

empty = [t for t in tasks if not t["contexts"]]
print(Counter(a for t in empty for a in t["Answerability"]))
# {'UNANSWERABLE': 55, 'CONVERSATIONAL': 10}
```

An empty context list is meaningful in this release. Do not remove these rows as generic missing data. Define what groundedness means for an abstention or an answer containing no factual claims, and retain the reason whenever a score is undefined.
:::

(mtrag-human)=
## Open the Human Ratings

The human evaluation JSON has seven top-level fields: `name`, `filters`, `models`, `metrics`, `documents`, `tasks` and `evaluations`. It is **not** a list of generation tasks. The `tasks` array refers to passage IDs; passage text lives in the separate `documents` array. Candidate text and its ratings live in `evaluations`.

| Object | Inspected size | Join or interpretation |
| :--- | ---: | :--- |
| `tasks` | 159 | Join to generation tasks on `task_id` |
| Represented conversations | 20 | Recovered through the generation-task join |
| `documents` | 350 | Join `contexts[].document_id` to `document_id` |
| `models` | 3 | Reference, Llama 3.1 405B, GPT-4o |
| `evaluations` | 477 | One record per task and response source |
| `metrics` | 12 | Mix of human and automatic metric definitions |

### Preserve the Raters

| Human metric | Recorded scale | Available individual ratings |
| :--- | :--- | ---: |
| Naturalness | Ordered categories 1–4 | 1,413 |
| Appropriateness | Ordered categories 1–4 | 1,413 |
| Completeness | Ordered categories 1–4 | 1,411 |
| Faithfulness | Ordered categories 1–4 | 1,419 |
| Win rate | Numeric 0–100 | 1,431 |

These are counts of annotation entries across the 477 evaluation records. They are **not** counts of distinct answers. The four category scales run from “No” to “Yes,” and the released metadata specifies median aggregation. Retain the original annotations before choosing a different analysis or aggregation rule.

:::{dropdown} Join the human subset and inspect one evaluation

Continue after the earlier loading example:

```python
human_path = cache / (
    "mtrag-human__evaluations__"
    "reference_subset_with_human_evaluations.json"
)
with human_path.open(encoding="utf-8") as stream:
    human = json.load(stream)

documents = {d["document_id"]: d for d in human["documents"]}
subset = {t["task_id"]: t for t in human["tasks"]}
evaluation = human["evaluations"][0]
source_task = by_id[evaluation["task_id"]]
subset_task = subset[evaluation["task_id"]]

print(source_task["conversation_id"], int(source_task["turn"]))
print(evaluation["model_id"])
print(evaluation["model_response"])
print(evaluation["annotations"]["faithfulness"])
for context in subset_task["contexts"]:
    print(documents[context["document_id"]]["text"])
```

Each human metric maps an annotator identifier to a record containing its value and metadata. The script flattens these into one row per **task × model × metric × annotator**, recovering conversation identity from the full generation file. Human metric names are selected from `metrics` where `author == "human"`, rather than guessing from the annotation keys.
:::

:::{admonition} Missing ratings are not zero ratings
:class: important

Three raters for each of 477 responses would give 1,431 entries per metric. Several metrics have fewer. Keep their actual denominators and missingness; never fill missing entries with zero. For paired human–judge analysis, join on the same task, candidate response and metric, then define how human ratings become the target quantity.
:::

(mtrag-research)=
## Connect to the Research

The proposal distinguishes variation in scores, overlap between metrics and the need for human labels. MTRAG supplies useful inputs, but does not by itself supply every experimental crossing needed for those questions.

::::{grid} 1 1 2 2

:::{grid-item-card} Part A · Reliability
Hold each candidate answer fixed. Cross selected judges, prompts and repeated calls, preserving conversation clusters. The public evaluations do not by themselves identify a new judge × prompt × run variance decomposition.
:::

:::{grid-item-card} Part A · Self-preference
Store candidate and judge model families explicitly. Include matching and non-matching combinations; control for answer quality and judge severity. A high score from one matching family alone is insufficient evidence.
:::

:::{grid-item-card} Part B · Metric redundancy
Align metrics on the same answer and evidence setting. Split by conversation before fitting reconstruction models. Keep all turns of a conversation together so shared history does not leak across folds.
:::

:::{grid-item-card} Part C · Human labels
The public human subset can support a pilot assessment of paired errors. Check its coverage and target population, align score scales and account for the 20 conversation clusters before extrapolating label requirements.
:::
::::

### A Proposed Analysis Table

This is a recommended output schema for the research pipeline, **not an existing MTRAG file**.

| Column group | Suggested columns | Purpose |
| :--- | :--- | :--- |
| Source | `release`, `setting`, `collection` | Identify data version and evidence conditions |
| Task | `conversation_id`, `task_id`, `turn` | Preserve the sampling hierarchy |
| Candidate | `agent_id`, `agent_family`, `answer_id` | Identify the exact fixed answer |
| Judge | `judge_id`, `judge_family`, `prompt_id`, `run_id` | Identify the evaluation condition |
| Measurement | `metric`, `score`, `score_scale`, `missing_reason` | Preserve meaning and undefined scores |
| Human link | `annotator_id`, `human_value` in a separate rating table | Retain individual human judgments |

:::{dropdown} A small first experiment

1. Select a few **whole conversations** from each collection, including unanswerable cases and later turns. Save the selected IDs and selection rule.
2. Begin with one evidence setting. Generate candidate answers once and retain the exact inputs, model version and outputs.
3. Choose a small set of clearly defined metrics. Specify reference use, score scales and abstention handling before judging.
4. Score the same answers under the intended judge, prompt and repetition combinations. Preserve unsuccessful calls as missing outcomes with reasons.
5. Check the resulting table for duplicate keys, incomplete crossings, score-range violations and unequal coverage before estimating reliability.

This is a pilot design recommendation. The page's inspection did not generate answers, call judges or estimate reliability coefficients.
:::

(mtrag-sources)=
## Sources and Next Steps

The next useful artifact is an answer-and-score table built from a documented pilot. This inspection establishes the file structure and valid joins that table will depend on.

| Source | Role in this page |
| :--- | :--- |
| [Research proposal](Proposal.ipynb) | Research questions and intended use of MTRAG |
| [MTRAG paper](https://aclanthology.org/2025.tacl-1.36/) | Benchmark motivation and creation process |
| [Human data documentation](https://github.com/IBM/mt-rag-benchmark/blob/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human/README.md) | File organization and review process |
| [Pinned generation files](https://github.com/IBM/mt-rag-benchmark/tree/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human/generation_tasks) | Actual task schemas, examples and computed distributions |
| [Pinned human evaluation file](https://github.com/IBM/mt-rag-benchmark/blob/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human/evaluations/reference_subset_with_human_evaluations.json) | Actual response sources, ratings, scales and joins |

:::{dropdown} Reproducibility record

**Repository:** IBM/mt-rag-benchmark  
**Commit:** `53fef6f5917c7e10c436f697e69a955dbc795d81`  
**Inspection date:** October 23, 2026

The three generation JSONL files, conversation JSON and human evaluation JSON were downloaded and parsed. The inspection script's assertions passed for task uniqueness, conversation–turn uniqueness, message order and length, setting membership, human-task joins, document joins and task–model evaluation uniqueness. Per-file hashes and byte counts are available in the downloadable summary.

Full corpus sizes are documented source counts. No model performance, metric validity or population-level reliability is established by these structural checks. Public availability also does not replace the repository's applicable license and citation requirements when redistributing data.
:::

::::::
